# 03 — Thai Transcript Extraction จากวิดีโอ

Notebook นี้ต่อจากขั้น Face Detection + Mouth Extraction

เป้าหมายคือสร้าง **ข้อความภาษาไทย + timestamp** จากวิดีโอ เพื่อเตรียมทำ label สำหรับ Visual Speech Recognition (Lip Reading)

สิ่งที่ได้:
- `transcript_segments.csv`
- `transcript_words.csv`
- `transcript_with_frames.csv`
- `transcript.txt`

> transcript จาก Speech-to-Text เป็น **weak label** ควรตรวจแก้ก่อนใช้ train supervised model จริง

## Cell 1 — ตั้ง Cache ไป Drive D

In [1]:
import os
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
CACHE_ROOT = PROJECT_ROOT / "cache"

HF_HOME = CACHE_ROOT / "huggingface"
TORCH_HOME = CACHE_ROOT / "torch"
TEMP_HOME = CACHE_ROOT / "temp"

for p in [HF_HOME, TORCH_HOME, TEMP_HOME]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TORCH_HOME"] = str(TORCH_HOME)
os.environ["TEMP"] = str(TEMP_HOME)
os.environ["TMP"] = str(TEMP_HOME)

print("HF_HOME   :", HF_HOME)
print("TORCH_HOME:", TORCH_HOME)
print("TEMP      :", TEMP_HOME)

HF_HOME   : D:\Face_detection\cache\huggingface
TORCH_HOME: D:\Face_detection\cache\torch
TEMP      : D:\Face_detection\cache\temp


## Cell 2 — ตรวจ Python / CUDA

In [2]:
import sys
import torch

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)
print("Python :", sys.executable)
print("Version:", sys.version.split()[0])
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("GPU    : ไม่พบ CUDA — ใช้ CPU ได้แต่จะช้ากว่า")

ENVIRONMENT CHECK
Python : c:\Users\ASUS FA507NUR\AppData\Local\Programs\Python\Python312\python.exe
Version: 3.12.9
Torch  : 2.6.0+cu124
CUDA   : True
GPU    : NVIDIA GeForce RTX 4050 Laptop GPU


## Cell 3 — ติดตั้ง packages

In [3]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "faster-whisper",
    "pandas",
    "tqdm"
])

print("Dependencies ready")

Dependencies ready


## Cell 4 — ตรวจ FFmpeg

In [6]:
import shutil

ffmpeg_path = shutil.which("ffmpeg")

if ffmpeg_path:
    print("System FFmpeg found:", ffmpeg_path)
else:
    print("System FFmpeg not found — this is OK.")
    print("faster-whisper will use PyAV for decoding.")

System FFmpeg not found — this is OK.
faster-whisper will use PyAV for decoding.


## Cell 5 — เลือกวิดีโอ

In [7]:
VIDEO_ID = "video001"

RAW_VIDEO_DIR = PROJECT_ROOT / "dataset" / "raw_videos"
VIDEO_PATH = RAW_VIDEO_DIR / f"{VIDEO_ID}.webm"

TRANSCRIPT_ROOT = PROJECT_ROOT / "dataset" / "transcripts" / VIDEO_ID
TRANSCRIPT_ROOT.mkdir(parents=True, exist_ok=True)

print("VIDEO_PATH      :", VIDEO_PATH)
print("TRANSCRIPT_ROOT :", TRANSCRIPT_ROOT)
print("Video exists    :", VIDEO_PATH.exists())

if not VIDEO_PATH.exists():
    raise FileNotFoundError(VIDEO_PATH)

VIDEO_PATH      : D:\Face_detection\dataset\raw_videos\video001.webm
TRANSCRIPT_ROOT : D:\Face_detection\dataset\transcripts\video001
Video exists    : True


## Cell 6 — อ่าน FPS ของวิดีโอ

In [8]:
import cv2

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError(f"เปิดวิดีโอไม่ได้: {VIDEO_PATH}")

FPS = float(cap.get(cv2.CAP_PROP_FPS))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

DURATION = TOTAL_FRAMES / FPS if FPS > 0 else 0

print("FPS         :", FPS)
print("Frames      :", TOTAL_FRAMES)
print("Resolution  :", f"{WIDTH}x{HEIGHT}")
print("Duration(s) :", round(DURATION, 2))

FPS         : 29.97002997002997
Frames      : 178335
Resolution  : 1920x1080
Duration(s) : 5950.44


## Cell 7 — โหลด Whisper Model

In [9]:
from faster_whisper import WhisperModel

MODEL_SIZE = "medium"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print("Model       :", MODEL_SIZE)
print("Device      :", DEVICE)
print("Compute type:", COMPUTE_TYPE)

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    download_root=str(HF_HOME / "faster_whisper")
)

print("Whisper model ready")

c:\Users\ASUS FA507NUR\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model       : medium
Device      : cuda
Compute type: float16


Whisper model ready


## Cell 8 — ถอดเสียงภาษาไทย

In [10]:
segments_generator, info = model.transcribe(
    str(VIDEO_PATH),
    language="th",
    beam_size=5,
    vad_filter=True,
    vad_parameters=dict(
        min_silence_duration_ms=400
    ),
    word_timestamps=True,
    condition_on_previous_text=True
)

print("Detected language:", info.language)
print("Language probability:", getattr(info, "language_probability", None))

segments = list(segments_generator)

print("Segments:", len(segments))

Detected language: th
Language probability: 1
Segments: 1293


## Cell 9 — แปลงผลเป็น DataFrame

In [11]:
import pandas as pd
import math

segment_rows = []
word_rows = []

for seg_index, seg in enumerate(segments):
    text = (seg.text or "").strip()

    segment_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": seg_index,
        "start_time": round(float(seg.start), 3),
        "end_time": round(float(seg.end), 3),
        "duration": round(float(seg.end - seg.start), 3),
        "text": text,
    })

    if seg.words:
        for word_index, word in enumerate(seg.words):
            word_rows.append({
                "video_id": VIDEO_ID,
                "segment_id": seg_index,
                "word_index": word_index,
                "start_time": round(float(word.start), 3),
                "end_time": round(float(word.end), 3),
                "word": (word.word or "").strip(),
                "probability": (
                    round(float(word.probability), 6)
                    if word.probability is not None
                    else None
                ),
            })

segments_df = pd.DataFrame(segment_rows)
words_df = pd.DataFrame(word_rows)

print("Segment rows:", len(segments_df))
print("Word rows   :", len(words_df))
display(segments_df.head(20))

Segment rows: 1293
Word rows   : 32976


,video_id,segment_id,start_time,end_time,duration,text
0,video001,0,0.42,2.20,1.78,และนี่คือ TiaMai Podcast
1,video001,1,2.48,4.44,1.96,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน
2,video001,2,4.44,6.50,2.06,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง
3,video001,3,6.50,7.94,1.44,ใช่ค่ะ และทั้งหมดนะครับ
4,video001,4,7.94,9.74,1.80,เป็นเพลงแค่ความคิดเห็นของเราสองคน
5,video001,5,9.74,11.90,2.16,ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ
6,video001,6,11.90,15.20,3.30,ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้...
7,video001,7,15.20,16.04,0.84,ใช่ค่ะ
8,video001,8,20.26,22.40,2.14,และวันนี้อย่างที่ทุกคนเห็นนะคะ
9,video001,9,22.40,50.76,28.36,นี่ๆ


## Cell 10 — ทำความสะอาดข้อความ

In [12]:
import re

def clean_thai_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

segments_df["text_raw"] = segments_df["text"]
segments_df["text"] = segments_df["text"].apply(clean_thai_text)

if len(words_df) > 0:
    words_df["word_raw"] = words_df["word"]
    words_df["word"] = words_df["word"].apply(clean_thai_text)

display(segments_df.head(20))

,video_id,segment_id,start_time,end_time,duration,text,text_raw
0,video001,0,0.42,2.20,1.78,และนี่คือ TiaMai Podcast,และนี่คือ TiaMai Podcast
1,video001,1,2.48,4.44,1.96,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน
2,video001,2,4.44,6.50,2.06,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง
3,video001,3,6.50,7.94,1.44,ใช่ค่ะ และทั้งหมดนะครับ,ใช่ค่ะ และทั้งหมดนะครับ
4,video001,4,7.94,9.74,1.80,เป็นเพลงแค่ความคิดเห็นของเราสองคน,เป็นเพลงแค่ความคิดเห็นของเราสองคน
5,video001,5,9.74,11.90,2.16,ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ,ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ
6,video001,6,11.90,15.20,3.30,ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้...,ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้...
7,video001,7,15.20,16.04,0.84,ใช่ค่ะ,ใช่ค่ะ
8,video001,8,20.26,22.40,2.14,และวันนี้อย่างที่ทุกคนเห็นนะคะ,และวันนี้อย่างที่ทุกคนเห็นนะคะ
9,video001,9,22.40,50.76,28.36,นี่ๆ,นี่ๆ


## Cell 11 — แปลง Timestamp เป็น Frame Index

In [13]:
segments_df["start_frame"] = (
    segments_df["start_time"] * FPS
).apply(math.floor).astype(int)

segments_df["end_frame"] = (
    segments_df["end_time"] * FPS
).apply(math.ceil).astype(int)

segments_df["frame_count"] = (
    segments_df["end_frame"]
    - segments_df["start_frame"]
    + 1
)

if len(words_df) > 0:
    words_df["start_frame"] = (
        words_df["start_time"] * FPS
    ).apply(math.floor).astype(int)

    words_df["end_frame"] = (
        words_df["end_time"] * FPS
    ).apply(math.ceil).astype(int)

display(
    segments_df[
        [
            "segment_id",
            "start_time",
            "end_time",
            "start_frame",
            "end_frame",
            "frame_count",
            "text",
        ]
    ].head(20)
)

,segment_id,start_time,end_time,start_frame,end_frame,frame_count,text
0,0,0.42,2.20,12,66,55,และนี่คือ TiaMai Podcast
1,1,2.48,4.44,74,134,61,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน
2,2,4.44,6.50,133,195,63,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง
3,3,6.50,7.94,194,238,45,ใช่ค่ะ และทั้งหมดนะครับ
4,4,7.94,9.74,237,292,56,เป็นเพลงแค่ความคิดเห็นของเราสองคน
5,5,9.74,11.90,291,357,67,ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ
6,6,11.90,15.20,356,456,101,ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้...
7,7,15.20,16.04,455,481,27,ใช่ค่ะ
8,8,20.26,22.40,607,672,66,และวันนี้อย่างที่ทุกคนเห็นนะคะ
9,9,22.40,50.76,671,1522,852,นี่ๆ


## Cell 12 — Export CSV / TXT

In [14]:
SEGMENTS_CSV = TRANSCRIPT_ROOT / "transcript_segments.csv"
WORDS_CSV = TRANSCRIPT_ROOT / "transcript_words.csv"
FRAMES_CSV = TRANSCRIPT_ROOT / "transcript_with_frames.csv"
TXT_PATH = TRANSCRIPT_ROOT / "transcript.txt"

segments_df.to_csv(
    SEGMENTS_CSV,
    index=False,
    encoding="utf-8-sig"
)

words_df.to_csv(
    WORDS_CSV,
    index=False,
    encoding="utf-8-sig"
)

segments_df[
    [
        "video_id",
        "segment_id",
        "start_time",
        "end_time",
        "start_frame",
        "end_frame",
        "frame_count",
        "text",
    ]
].to_csv(
    FRAMES_CSV,
    index=False,
    encoding="utf-8-sig"
)

with open(TXT_PATH, "w", encoding="utf-8") as f:
    for _, row in segments_df.iterrows():
        f.write(
            f"[{row['start_time']:.3f} - "
            f"{row['end_time']:.3f}] "
            f"{row['text']}\n"
        )

print("Saved:")
print(" -", SEGMENTS_CSV)
print(" -", WORDS_CSV)
print(" -", FRAMES_CSV)
print(" -", TXT_PATH)

Saved:
 - D:\Face_detection\dataset\transcripts\video001\transcript_segments.csv
 - D:\Face_detection\dataset\transcripts\video001\transcript_words.csv
 - D:\Face_detection\dataset\transcripts\video001\transcript_with_frames.csv
 - D:\Face_detection\dataset\transcripts\video001\transcript.txt


## Cell 13 — Preview Transcript

In [15]:
for _, row in segments_df.head(50).iterrows():
    print(
        f"[{row['start_time']:8.3f} → "
        f"{row['end_time']:8.3f}] "
        f"{row['text']}"
    )

[   0.420 →    2.200] และนี่คือ TiaMai Podcast
[   2.480 →    4.440] Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน
[   4.440 →    6.500] ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง
[   6.500 →    7.940] ใช่ค่ะ และทั้งหมดนะครับ
[   7.940 →    9.740] เป็นเพลงแค่ความคิดเห็นของเราสองคน
[   9.740 →   11.900] ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ
[  11.900 →   15.200] ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้เลยนะคะ
[  15.200 →   16.040] ใช่ค่ะ
[  20.260 →   22.400] และวันนี้อย่างที่ทุกคนเห็นนะคะ
[  22.400 →   50.760] นี่ๆ
[  50.760 →   52.780] เพราะปฏิเสธไม่ได้เลยว่าการรู้ภาษาอังกฤษนี้
[  52.780 →   55.400] ก็จะช่วยเปิดช่องทางเราในหลายๆด้าน
[  55.400 →   56.920] อย่างเช่นการเอาไปอับเงินเดือน
[  56.920 →   58.760] หรือการเติบโตในหน้าที่การงาน
[  58.760 →   62.140] เพราะฉะนั้นวันนี้นะคะ สปอนเซอร์ของเราก็จะเป็น
[  62.620 →   63.840] Elsa Fleek
[  67.580 →   69.540] เขาจะเป็นアプลิเคชั่นเรียนภาษาอังกฤษ
[  69.540 →   72.020] เหมือนเรามีครูส่วนตัวที่ไปทุกที่กับเรา
[  72.020 →   74.500] เรียนได้แบบ 24 ชั่วโมงเลย

## Cell 14 — ตรวจ overlap กับ Mouth Metadata

In [16]:
MOUTH_METADATA_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "mouth_crops"
    / VIDEO_ID
    / "mouth_metadata.csv"
)

print("Mouth metadata exists:", MOUTH_METADATA_PATH.exists())

overlap_rows = []

if MOUTH_METADATA_PATH.exists():

    mouth_df = pd.read_csv(MOUTH_METADATA_PATH)
    mouth_df = mouth_df[
        mouth_df["accepted"] == True
    ].copy()

    for _, seg in segments_df.iterrows():

        matched = mouth_df[
            (mouth_df["frame_index"] >= seg["start_frame"])
            &
            (mouth_df["frame_index"] <= seg["end_frame"])
        ]

        counts = (
            matched
            .groupby("track_id")["frame_index"]
            .count()
            .to_dict()
        )

        overlap_rows.append({
            "segment_id": int(seg["segment_id"]),
            "start_time": seg["start_time"],
            "end_time": seg["end_time"],
            "start_frame": int(seg["start_frame"]),
            "end_frame": int(seg["end_frame"]),
            "text": seg["text"],
            "mouth_frames_by_track": counts,
        })

    overlap_df = pd.DataFrame(overlap_rows)
    display(overlap_df.head(30))

else:
    print("ยังไม่มี mouth_metadata.csv")

Mouth metadata exists: True


,segment_id,start_time,end_time,start_frame,end_frame,text,mouth_frames_by_track
0,0,0.42,2.20,12,66,และนี่คือ TiaMai Podcast,"{0: 55, 1: 55}"
1,1,2.48,4.44,74,134,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน,"{0: 26, 1: 26}"
2,2,4.44,6.50,133,195,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง,{}
3,3,6.50,7.94,194,238,ใช่ค่ะ และทั้งหมดนะครับ,{}
4,4,7.94,9.74,237,292,เป็นเพลงแค่ความคิดเห็นของเราสองคน,{}
5,5,9.74,11.90,291,357,ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ,{}
6,6,11.90,15.20,356,456,ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้...,{}
7,7,15.20,16.04,455,481,ใช่ค่ะ,{}
8,8,20.26,22.40,607,672,และวันนี้อย่างที่ทุกคนเห็นนะคะ,{}
9,9,22.40,50.76,671,1522,นี่ๆ,{}


## Cell 15 — Export Transcript ↔ Mouth Overlap

In [17]:
if "overlap_df" in globals() and len(overlap_df) > 0:

    OVERLAP_CSV = (
        TRANSCRIPT_ROOT
        / "transcript_mouth_overlap.csv"
    )

    overlap_df.to_csv(
        OVERLAP_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", OVERLAP_CSV)

else:
    print("ยังไม่มี overlap data")

Saved: D:\Face_detection\dataset\transcripts\video001\transcript_mouth_overlap.csv


# ขั้นต่อไป

หลัง Notebook นี้ผ่าน เราจะมี:

```text
video001.webm
├── Mouth 96×96
└── Transcript + timestamp
```

ขั้นถัดไปคือ **Speaker / Track Alignment** เพื่อหาว่า transcript แต่ละช่วงเป็นของ `track_000` หรือ `track_001`

จากนั้นจึงสร้าง labeled lip clips สำหรับ train